In [ ]:
import sqlite3
import json
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup


In [ ]:
conn = sqlite3.connect("/content/db")

members = pd.read_sql("SELECT * FROM members", conn)
books = pd.read_sql("SELECT * FROM books", conn)
checkouts = pd.read_sql("SELECT * FROM checkouts", conn)



In [ ]:
print(f"members: {members.shape[0]} rows")
print(f"books:  {books.shape[0]} rows")
print(f"checkouts:  {checkouts.shape[0]} rows")

members: 80 rows
books:  32 rows
checkouts:  391 rows


In [ ]:
members.head()

,member_id,first_name,last_name,grade,neighborhood,membership_status,join_date
0,1001,Salma,Ibrahim,8.0,Maadi,Active,2023-04-05
1,1002,Fares,Saleh,9.0,Maadi,Active,None
2,1003,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23
3,1004,Fares,Wahba,7.0,Maadi,inactive,2024-10-09
4,1005,Youssef,Halim,9.0,Maadi,Active,2024-05-05


In [ ]:
books.head()

,book_id,title,author
0,501,The Silver Kite,Amina Darwish
1,502,Desert Compass,Amina Darwish
2,503,The Lantern Maker,Adel Roushdy
3,504,Rooftop Astronomers,Adel Roushdy
4,505,Letters to the Nile,Aya Hafez


In [ ]:
checkouts.head()

,checkout_id,member_id,book_id,checkout_date,return_date
0,9263,1047,517,2024-10-21,2024-11-07
1,9340,1072,513,2025-08-24,2025-09-01
2,9231,1053,523,2024-02-04,2024-02-16
3,9129,1032,513,2025-06-21,2025-06-29
4,9370,1079,511,2025-11-11,2025-12-03


In [ ]:
def run(query):
  return pd.read_sql_query(query,conn)

In [ ]:
#Q1:How much is each member borrowing?
query='''SELECT
      m.first_name || ' ' || m.last_name AS member_name,
       COUNT(c.checkout_id) AS total_checkouts
       FROM members m
       LEFT JOIN checkouts c
       ON m.member_id = c.member_id
      GROUP BY
      m.first_name,
      m.last_name
    '''
print("each student number of books borrowed: ")
run(query)

each student number of books borrowed: 


,member_name,total_checkouts
0,Adam Badr,2
1,Adam Fahmy,17
2,Adam Shafik,5
3,Ahmed Fahmy,10
4,Ahmed Gamal,1
...,...,...
75,Youssef Hegazy,17
76,Youssef Kamel,0
77,Ziad Fahmy,8
78,Ziad Fouad,2


In [ ]:
#Q2:Which books match a chosen author pattern?
query='''
SELECT title FROM books
WHERE title LIKE '%on%'
'''
print("books contain 'on' : ")
run(query)

books contain 'on' : 


,title
0,Rooftop Astronomers
1,The Missing Metronome
2,Songs of the Oasis
3,Shadows on the Corniche
4,A Garden of Equations
5,The Sandstone Key


In [ ]:
#Q3:What are the most popular books?"FRIST 5"
query='''SELECT b.title ,
COUNT(c.checkout_id) AS total_checkouts
FROM books b
LEFT JOIN checkouts c ON b.book_id = c.book_id
GROUP BY b.title
ORDER BY
total_checkouts DESC
LIMIT 5'''
print("top 5 borrowed books:")
run(query)

top 5 borrowed books:


,title,total_checkouts
0,The Silver Kite,57
1,Fossils and Fireflies,55
2,Circuits for Beginners,46
3,Kites Over Cairo,38
4,Storms and Sailboats,25


In [ ]:
#Q4:4. Who are the most active readers?
query='''SELECT m.first_name || ' ' || m.last_name AS member_name,
         COUNT(c.checkout_id) AS BOROWS
         FROM members m
         LEFT JOIN checkouts c
         ON m.member_id = c.member_id
         GROUP BY
         m.first_name,
         m.last_name
         ORDER BY
         BOROWS DESC
         LIMIT 10'''
print("top 10 readers:")
run(query)

top 10 readers:


,member_name,BOROWS
0,Aya Wahba,25
1,Sherif Saleh,21
2,Ziad Saleh,19
3,Mostafa Fouad,18
4,Nour Nabil,18
5,Adam Fahmy,17
6,Ahmed Shafik,17
7,Youssef Hegazy,17
8,Reem Osman,16
9,Sara Rashad,16


In [ ]:
#Q5:What does a neighborhood's activity look like further back in time?
query='''SELECT
m.first_name || ' ' || m.last_name AS member_name,
         COUNT(c.checkout_id) AS BOROWS
         FROM members m
         JOIN checkouts c
         ON m.member_id = c.member_id
         WHERE m.neighborhood ='Maadi'
         GROUP BY
         m.member_id, m.first_name, m.last_name
         ORDER BY
         c.checkout_date DESC
         limit 10
         OFFSET 10
         '''
print("readers actvity in Maadi neighborhood:")
run(query)

readers actvity in Maadi neighborhood:


,member_name,BOROWS
0,Adam Badr,2
1,Menna Halim,2
2,Salma Ibrahim,1
3,Youssef Fouad,6
4,Dina Rashad,7
5,Habiba Nabil,1
6,Fares Saleh,2
7,Youssef Halim,3


In [ ]:
books.head()

,book_id,title,author
0,501,The Silver Kite,Amina Darwish
1,502,Desert Compass,Amina Darwish
2,503,The Lantern Maker,Adel Roushdy
3,504,Rooftop Astronomers,Adel Roushdy
4,505,Letters to the Nile,Aya Hafez


In [ ]:

query='''SELECT *
FROM checkouts c
LEFT JOIN books b
ON b.book_id = c.book_id;
'''
run(query)

,checkout_id,member_id,book_id,checkout_date,return_date,book_id,title,author
0,9263,1047,517,2024-10-21,2024-11-07,517,Shadows on the Corniche,Hani Nagati
1,9340,1072,513,2025-08-24,2025-09-01,513,Circuits for Beginners,Galal Mounir
2,9231,1053,523,2024-02-04,2024-02-16,523,Footsteps in the Dust,Laila Shokry
3,9129,1032,513,2025-06-21,2025-06-29,513,Circuits for Beginners,Galal Mounir
4,9370,1079,511,2025-11-11,2025-12-03,511,Winter in Alexandria,Farida Anwar
...,...,...,...,...,...,...,...,...
386,9232,1044,513,2025-05-26,2025-06-11,513,Circuits for Beginners,Galal Mounir
387,9084,1008,511,2024-06-27,2024-07-22,511,Winter in Alexandria,Farida Anwar
388,9116,1024,519,2025-01-10,2025-02-09,519,Kites Over Cairo,Jasmine Wahdan
389,9352,1076,501,2025-02-02,None,501,The Silver Kite,Amina Darwish


In [ ]:
stage1 = checkouts.merge(
    members,
    on="member_id",
    how="left")
print(stage1.shape)
stage1.head()

(391, 11)


,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date
0,9263,1047,517,2024-10-21,2024-11-07,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25
1,9340,1072,513,2025-08-24,2025-09-01,Seif,Zaki,9.0,Zamalek,Active,2025-10-21
2,9231,1053,523,2024-02-04,2024-02-16,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03
3,9129,1032,513,2025-06-21,2025-06-29,Nada,Zaki,7.0,Nasr City,Active,2025-10-19
4,9370,1079,511,2025-11-11,2025-12-03,Rana,Osman,8.0,Shubra,Active,2024-10-27


In [ ]:
stage2 = stage1.merge(
    books,
    on="book_id",
    how="left")
print(stage2.shape)
stage2.head()

(391, 13)


,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,title,author
0,9263,1047,517,2024-10-21,2024-11-07,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25,Shadows on the Corniche,Hani Nagati
1,9340,1072,513,2025-08-24,2025-09-01,Seif,Zaki,9.0,Zamalek,Active,2025-10-21,Circuits for Beginners,Galal Mounir
2,9231,1053,523,2024-02-04,2024-02-16,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03,Footsteps in the Dust,Laila Shokry
3,9129,1032,513,2025-06-21,2025-06-29,Nada,Zaki,7.0,Nasr City,Active,2025-10-19,Circuits for Beginners,Galal Mounir
4,9370,1079,511,2025-11-11,2025-12-03,Rana,Osman,8.0,Shubra,Active,2024-10-27,Winter in Alexandria,Farida Anwar


In [ ]:
json_data = pd.read_json("/content/json")

In [ ]:
json_data.head()

,book_id,genre,pages,publication_year,publisher
0,501,Adventure,128,2017.0,Nile Press
1,502,Adventure,109,2018.0,Delta House
2,503,Historical,259,NaN,Nile Press
3,504,Science,319,2009.0,Cairo Young Readers
4,505,Historical,216,2024.0,Oasis Books


In [ ]:
stage3 = stage2.merge(
    json_data,
    on="book_id",
    how="left")
print(stage3.shape)
stage3.head()

(391, 17)


,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,title,author,genre,pages,publication_year,publisher
0,9263,1047,517,2024-10-21,2024-11-07,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25,Shadows on the Corniche,Hani Nagati,Mystery,338,2015.0,Delta House
1,9340,1072,513,2025-08-24,2025-09-01,Seif,Zaki,9.0,Zamalek,Active,2025-10-21,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books
2,9231,1053,523,2024-02-04,2024-02-16,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03,Footsteps in the Dust,Laila Shokry,Historical,276,2018.0,Oasis Books
3,9129,1032,513,2025-06-21,2025-06-29,Nada,Zaki,7.0,Nasr City,Active,2025-10-19,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books
4,9370,1079,511,2025-11-11,2025-12-03,Rana,Osman,8.0,Shubra,Active,2024-10-27,Winter in Alexandria,Farida Anwar,Historical,117,2016.0,Nile Press


In [ ]:
html_tables = pd.read_html("/content/html")

reading_kickoff = html_tables[0]#to take the table
print(reading_kickoff.shape)
reading_kickoff.head()

(26, 3)


,Member ID,Book ID,Checkout Date
0,1026,522,2025-07-11
1,1049,520,2025-07-11
2,1062,525,2025-07-05
3,1065,520,2025-07-07
4,1104,515,2025-07-07


In [ ]:
#change coulmn names from capital to small to be able to be added
reading_kickoff = reading_kickoff.rename(columns={
    'Member ID': 'member_id',
    'Book ID': 'book_id',
    'Checkout Date': 'checkout_date'
})

reading_kickoff.head()

,member_id,book_id,checkout_date
0,1026,522,2025-07-11
1,1049,520,2025-07-11
2,1062,525,2025-07-05
3,1065,520,2025-07-07
4,1104,515,2025-07-07


In [ ]:

kickoff_stage1 = reading_kickoff.merge(members, on='member_id', how='left')
kickoff_stage1.shape

(26, 9)

In [ ]:
kickoff_stage2 = kickoff_stage1.merge(books, on='book_id', how='left')
kickoff_stage3 = kickoff_stage2.merge(json_data, on='book_id', how='left')

In [ ]:
final_dataset = pd.concat([stage3, kickoff_stage3], ignore_index=True)
final_dataset.shape

(417, 17)

In [ ]:
final_dataset.tail(26)

,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,title,author,genre,pages,publication_year,publisher
391,NaN,1026,522,2025-07-11,NaN,Nada,Saleh,7.0,Nasr City,inactive,2023-11-16,The Puzzle Merchant,Karim Elwy,Mystery,104,2016.0,Cairo Young Readers
392,NaN,1049,520,2025-07-11,NaN,Ahmed,Gamal,6.0,Heliopolis,Active,2023-06-10,The Copper Telescope,Jasmine Wahdan,Science Fiction,136,2009.0,Oasis Books
393,NaN,1062,525,2025-07-05,NaN,Tarek,Adel,7.0,Zamalek,Active,2023-01-22,Storms and Sailboats,Mahmoud Rafei,Adventure,297,2015.0,Oasis Books
394,NaN,1065,520,2025-07-07,NaN,Adam,Fahmy,6.0,Zamalek,Active,2025-07-12,The Copper Telescope,Jasmine Wahdan,Science Fiction,136,2009.0,Oasis Books
395,NaN,1104,515,2025-07-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Songs of the Oasis,Hoda Bakry,Poetry,316,2022.0,Cairo Young Readers
396,NaN,1009,503,2025-07-09,NaN,Hassan,Saleh,6.0,Maadi,Active,2025-04-25,The Lantern Maker,Adel Roushdy,Historical,259,NaN,Nile Press
397,NaN,1063,522,2025-07-07,NaN,Layla,Fouad,6.0,Zamalek,Active,2024-06-21,The Puzzle Merchant,Karim Elwy,Mystery,104,2016.0,Cairo Young Readers
398,NaN,1022,511,2025-07-12,NaN,Youssef,Fouad,7.0,Maadi,Active,2025-02-28,Winter in Alexandria,Farida Anwar,Historical,117,2016.0,Nile Press
399,NaN,1029,523,2025-07-09,NaN,Rana,Kamel,6.0,Nasr City,Active,2023-04-03,Footsteps in the Dust,Laila Shokry,Historical,276,2018.0,Oasis Books
400,NaN,1201,509,2025-07-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Marbles and Mirrors,Diaa Sultan,Mystery,221,2011.0,Nile Press


The Reading Kickoff data was available on a web page, and there was no API available to provide the data directly.so the webpage was the main source of data .so

In [ ]:
final_dataset.to_csv('task1_combined_data.csv', index=False)

In [ ]:
df= pd.read_csv('task1_combined_data.csv')

In [ ]:
df.head()


,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,title,author,genre,pages,publication_year,publisher
0,9263.0,1047,517,2024-10-21,2024-11-07,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25,Shadows on the Corniche,Hani Nagati,Mystery,338,2015.0,Delta House
1,9340.0,1072,513,2025-08-24,2025-09-01,Seif,Zaki,9.0,Zamalek,Active,2025-10-21,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books
2,9231.0,1053,523,2024-02-04,2024-02-16,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03,Footsteps in the Dust,Laila Shokry,Historical,276,2018.0,Oasis Books
3,9129.0,1032,513,2025-06-21,2025-06-29,Nada,Zaki,7.0,Nasr City,Active,2025-10-19,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books
4,9370.0,1079,511,2025-11-11,2025-12-03,Rana,Osman,8.0,Shubra,Active,2024-10-27,Winter in Alexandria,Farida Anwar,Historical,117,2016.0,Nile Press


In [ ]:
df.shape

(417, 17)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 417 entries, 0 to 416
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   checkout_id        391 non-null    float64
 1   member_id          417 non-null    int64  
 2   book_id            417 non-null    int64  
 3   checkout_date      417 non-null    object 
 4   return_date        326 non-null    object 
 5   first_name         412 non-null    object 
 6   last_name          412 non-null    object 
 7   grade              376 non-null    float64
 8   neighborhood       412 non-null    object 
 9   membership_status  412 non-null    object 
 10  join_date          406 non-null    object 
 11  title              417 non-null    object 
 12  author             417 non-null    object 
 13  genre              417 non-null    object 
 14  pages              417 non-null    int64  
 15  publication_year   382 non-null    float64
 16  publisher          417 non

In [ ]:
print(df['checkout_id'].duplicated().sum())

33


In [ ]:
df.duplicated().sum()

np.int64(8)

In [ ]:
df = df.drop_duplicates(keep='first')

In [ ]:
print(df['checkout_id'].duplicated().sum())

25


In [ ]:
print(df.isna().sum())

checkout_id          26
member_id             0
book_id               0
checkout_date         0
return_date          91
first_name            5
last_name             5
grade                41
neighborhood          5
membership_status     5
join_date            11
title                 0
author                0
genre                 0
pages                 0
publication_year     33
publisher             0
dtype: int64


In [ ]:
missing_ids = df['checkout_id'].isna()

start_id = int(df['checkout_id'].max()) + 1

df.loc[missing_ids, 'checkout_id'] = range(
    start_id,
    start_id + missing_ids.sum()
)

In [ ]:
print(df.isna().sum())

checkout_id           0
member_id             0
book_id               0
checkout_date         0
return_date          91
first_name            5
last_name             5
grade                41
neighborhood          5
membership_status     5
join_date            11
title                 0
author                0
genre                 0
pages                 0
publication_year     33
publisher             0
dtype: int64


In [ ]:
df.dropna(subset=['first_name'], inplace=True)

In [ ]:
print(df.isna().sum())

checkout_id           0
member_id             0
book_id               0
checkout_date         0
return_date          86
first_name            0
last_name             0
grade                36
neighborhood          0
membership_status     0
join_date             6
title                 0
author                0
genre                 0
pages                 0
publication_year     33
publisher             0
dtype: int64


In [ ]:
#fill grade mising values with median
mid1=df["grade"].median()
df["grade"]=df["grade"].fillna(mid1)

In [ ]:
print(df.isna().sum())

checkout_id           0
member_id             0
book_id               0
checkout_date         0
return_date          86
first_name            0
last_name             0
grade                 0
neighborhood          0
membership_status     0
join_date             6
title                 0
author                0
genre                 0
pages                 0
publication_year     33
publisher             0
dtype: int64


In [ ]:
print(df['publication_year'].mode())

0    2024.0
Name: publication_year, dtype: float64


In [ ]:
print(df['publication_year'].median())

2017.0


In [ ]:
print(df['publication_year'].mean())

2017.4393530997304


In [ ]:
#fill publication_year mising values with median
mid2=df["publication_year"].median()
df["publication_year"]=df["publication_year"].fillna(mid2)

In [ ]:
print(df.isna().sum())

checkout_id           0
member_id             0
book_id               0
checkout_date         0
return_date          86
first_name            0
last_name             0
grade                 0
neighborhood          0
membership_status     0
join_date             6
title                 0
author                0
genre                 0
pages                 0
publication_year      0
publisher             0
dtype: int64


In [ ]:
# Convert dates to datetime
df['return_date'] = pd.to_datetime(df['return_date'], errors='coerce')
df["return_date"]=df["return_date"].fillna("still active")
#i will Keep missing return dates as still active because they  represent books that have not been returned yet
#instead of deleting 86 rows

In [ ]:
print(df.isna().sum())

checkout_id          0
member_id            0
book_id              0
checkout_date        0
return_date          0
first_name           0
last_name            0
grade                0
neighborhood         0
membership_status    0
join_date            6
title                0
author               0
genre                0
pages                0
publication_year     0
publisher            0
dtype: int64


In [ ]:
df = df.dropna(subset=['join_date'])

In [ ]:
print(df.isna().sum())

checkout_id          0
member_id            0
book_id              0
checkout_date        0
return_date          0
first_name           0
last_name            0
grade                0
neighborhood         0
membership_status    0
join_date            0
title                0
author               0
genre                0
pages                0
publication_year     0
publisher            0
dtype: int64


In [ ]:
df.head(1)

,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,title,author,genre,pages,publication_year,publisher
0,9263.0,1047,517,2024-10-21,2024-11-07 00:00:00,Sara,Rashad,8.0,Heliopolis,Inactive,2024-06-25,Shadows on the Corniche,Hani Nagati,Mystery,338,2015.0,Delta House


In [ ]:
df['neighborhood'].value_counts()

,count
neighborhood,
Nasr City,97
Maadi,92
Heliopolis,86
Zamalek,58
Shubra,34
Maadi,19
zamalek,10
NASR CITY,1
HELIOPOLIS,1


In [ ]:
df['neighborhood'] = df['neighborhood'].str.strip().str.title()
df["neighborhood"].value_counts()

,count
neighborhood,
Maadi,111
Nasr City,98
Heliopolis,87
Zamalek,68
Shubra,34


In [ ]:
df['membership_status'].value_counts()

,count
membership_status,
Active,272
Inactive,52
active,43
inactive,31


In [ ]:
df['membership_status'] = df['membership_status'].str.strip().str.title()
df["membership_status"].value_counts()

,count
membership_status,
Active,315
Inactive,83


In [ ]:
df['title'] = df['title'].str.strip().str.title()
df["title"].value_counts()

,count
title,
The Silver Kite,54
Fossils And Fireflies,53
Circuits For Beginners,47
Kites Over Cairo,37
Storms And Sailboats,26
The Paper Boat Club,14
The Beekeeper'S Almanac,11
The Copper Telescope,10
The Puzzle Merchant,10


In [ ]:
df['author'] = df['author'].str.strip().str.title()
df["author"].value_counts()

,count
author,
Amina Darwish,61
Dalia Serry,54
Galal Mounir,53
Jasmine Wahdan,47
Mahmoud Rafei,30
Aya Hafez,22
Diaa Sultan,19
Hoda Bakry,16
Karim Elwy,16


In [ ]:
df['publisher'] = df['publisher'].str.strip().str.title()
df["publisher"].value_counts()

,count
publisher,
Nile Press,197
Oasis Books,110
Cairo Young Readers,51
Delta House,40


In [ ]:
# IQR-based outlier handling
Q1 = df['grade'].quantile(0.25)
Q3 = df['grade'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1: {Q1}")
print(f"Q3: {Q3}")
print(f"IQR: {IQR:.1f}")
print(f"Lower bound: {lower_bound:.1f}")
print(f"Upper bound: {upper_bound:.1f}")
print(f"Outliers above upper bound: {(df['grade'] > upper_bound).sum()}")

Q1: 7.0
Q3: 9.0
IQR: 2.0
Lower bound: 4.0
Upper bound: 12.0
Outliers above upper bound: 0


In [ ]:
int_columns = ['checkout_id', 'grade', 'publication_year']
for col in int_columns:
    df[col] = df[col].astype(int)

In [ ]:
df['checkout_date'] = pd.to_datetime(df['checkout_date']).dt.strftime('%Y-%m-%d')
df['join_date'] = pd.to_datetime(df['join_date']).dt.strftime('%Y-%m-%d')

df['return_date'] = df['return_date'].apply(
    lambda x: pd.to_datetime(x).strftime('%Y-%m-%d') if x != 'still active' else x
)

In [ ]:
df.head()

,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,title,author,genre,pages,publication_year,publisher
0,9263,1047,517,2024-10-21,2024-11-07,Sara,Rashad,8,Heliopolis,Inactive,2024-06-25,Shadows On The Corniche,Hani Nagati,Mystery,338,2015,Delta House
1,9340,1072,513,2025-08-24,2025-09-01,Seif,Zaki,9,Zamalek,Active,2025-10-21,Circuits For Beginners,Galal Mounir,Science,294,2021,Oasis Books
2,9231,1053,523,2024-02-04,2024-02-16,Adam,Shafik,9,Heliopolis,Active,2024-01-03,Footsteps In The Dust,Laila Shokry,Historical,276,2018,Oasis Books
3,9129,1032,513,2025-06-21,2025-06-29,Nada,Zaki,7,Nasr City,Active,2025-10-19,Circuits For Beginners,Galal Mounir,Science,294,2021,Oasis Books
4,9370,1079,511,2025-11-11,2025-12-03,Rana,Osman,8,Shubra,Active,2024-10-27,Winter In Alexandria,Farida Anwar,Historical,117,2016,Nile Press


I noticed that in row 2 that ***checkout_date*** is older than ***join_date***
🤔

In [ ]:
df.to_csv('task2_cleaned_data.csv', index=False)

In [ ]:
df = pd.read_csv('task2_cleaned_data.csv')
members = df.groupby('neighborhood')['member_id'].nunique()
checkouts = df.groupby('neighborhood')['checkout_id'].count()

In [ ]:

result = pd.DataFrame({'Members': members, 'Checkouts': checkouts})
print(result)

              Members  Checkouts
neighborhood                    
Heliopolis         13         87
Maadi              19        111
Nasr City          14         98
Shubra              5         34
Zamalek            11         68
